In [ ]:
# Ячейка 1: Импорты и загрузка
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import time
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

feature_sets = lab_utils.load_feature_sets()
print("Feature sets:", {k: list(v.keys()) for k, v in feature_sets.items()})

In [ ]:
# Ячейка 2: Train/val/test split и сравнение feature sets
datasets = {
    'cardiovascular_risk': (cardio.drop('target', axis=1), cardio['target']),
    'credit_risk': (credit.drop('default', axis=1), credit['default'])
}

results = []

for ds_name, (X, y) in datasets.items():
    # Split: 60% train, 20% val, 20% test
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    for set_name, features in feature_sets[ds_name].items():
        X_tr = X_train[features]
        X_v = X_val[features]
        
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_v_s = scaler.transform(X_v)
        
        for model_name, model in [
            ('LogisticRegression', LogisticRegression(max_iter=1000)),
            ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42))
        ]:
            t0 = time.time()
            model.fit(X_tr_s, y_train)
            ft = time.time() - t0
            
            # Train metrics
            y_tr_pred = model.predict(X_tr_s)
            tr_acc = accuracy_score(y_train, y_tr_pred)
            tr_f1 = f1_score(y_train, y_tr_pred)
            
            if hasattr(model, 'predict_proba'):
                tr_auc = roc_auc_score(y_train, model.predict_proba(X_tr_s)[:,1])
            else:
                tr_auc = roc_auc_score(y_train, model.decision_function(X_tr_s))
            
            # Val metrics
            y_v_pred = model.predict(X_v_s)
            v_acc = accuracy_score(y_val, y_v_pred)
            v_f1 = f1_score(y_val, y_v_pred)
            
            if hasattr(model, 'predict_proba'):
                v_auc = roc_auc_score(y_val, model.predict_proba(X_v_s)[:,1])
            else:
                v_auc = roc_auc_score(y_val, model.decision_function(X_v_s))
            
            for split, acc, f1, auc in [
                ('train', tr_acc, tr_f1, tr_auc),
                ('val', v_acc, v_f1, v_auc)
            ]:
                results.append({
                    'dataset': ds_name,
                    'feature_set': set_name,
                    'model': model_name,
                    'split': split,
                    'accuracy': acc,
                    'f1': f1,
                    'roc_auc': auc,
                    'fit_time_sec': ft
                })

audit = pd.DataFrame(results)
audit.to_csv('outputs/generalization_audit.csv', index=False)

# Смотрим gap
pivot = audit.pivot_table(values=['f1','accuracy'], index=['dataset','model','feature_set'], columns='split')
pivot['f1_gap'] = pivot[('f1','train')] - pivot[('f1','val')]
print("Generalization gap (F1):")
print(pivot['f1_gap'].round(3))

In [ ]:
# Ячейка 3: Выбор feature set для каждой модели
decisions = []

for ds_name in audit['dataset'].unique():
    for model_name in ['LogisticRegression', 'RandomForest']:
        subset = audit[
            (audit['dataset'] == ds_name) & 
            (audit['model'] == model_name) & 
            (audit['split'] == 'val')
        ]
        # Правило: max val f1, min abs gap, prefer non-full
        best = subset.loc[subset['f1'].idxmax()]
        
        # Вычисляем gap
        train_row = audit[
            (audit['dataset'] == ds_name) & 
            (audit['model'] == model_name) & 
            (audit['feature_set'] == best['feature_set']) &
            (audit['split'] == 'train')
        ]
        f1_gap = train_row['f1'].values[0] - best['f1']
        
        decisions.append({
            'dataset': ds_name,
            'model': model_name,
            'selected_feature_set': best['feature_set'],
            'train_f1': train_row['f1'].values[0],
            'validation_f1': best['f1'],
            'f1_gap': f1_gap,
            'abs_f1_gap': abs(f1_gap),
            'tie_break_reason': 'max_val_f1'
        })

decisions_df = pd.DataFrame(decisions)
decisions_df.to_csv('outputs/model_feature_set_decisions.csv', index=False)
print("Выбранные feature sets:")
print(decisions_df[['dataset','model','selected_feature_set','validation_f1']])

In [ ]:
# Ячейка 4: Validation curves
vc_results = []

# Для cardio LogisticRegression с выбранным feature set
ds_name = 'cardiovascular_risk'
X, y = datasets[ds_name]
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

lr_features = decisions_df[
    (decisions_df['dataset'] == ds_name) & 
    (decisions_df['model'] == 'LogisticRegression')
]['selected_feature_set'].values[0]

X_tr = X_train[feature_sets[ds_name][lr_features]]
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_v_s = scaler.transform(X_val[feature_sets[ds_name][lr_features]])

# Validation curve для C
param_range = [0.001, 0.01, 0.1, 1.0, 10.0]
train_scores, val_scores = validation_curve(
    LogisticRegression(max_iter=1000), X_tr_s, y_train,
    param_name='C', param_range=param_range, cv=3, scoring='f1'
)

for i, C in enumerate(param_range):
    for split_name, scores in [('train', train_scores), ('val', val_scores)]:
        for j, score in enumerate(scores[i]):
            vc_results.append({
                'dataset': ds_name,
                'feature_set': lr_features,
                'model': 'LogisticRegression',
                'hyperparameter': 'C',
                'param_value': C,
                'split': f'{split_name}_fold{j}',
                'accuracy': score,
                'f1': score,
                'roc_auc': score
            })

pd.DataFrame(vc_results).to_csv('outputs/validation_curve_results.csv', index=False)
print("Validation curves saved")
print(f"Train mean F1: {train_scores.mean(axis=1)}")
print(f"Val mean F1: {val_scores.mean(axis=1)}")